# UMLS lookup helper

Streams the 5 RRF tables on demand to assemble everything UMLS knows about a single CUI:

| Table | What it gives you |
|---|---|
| **MRCONSO** | All atom strings for the concept (preferred name, synonyms, source vocabularies) |
| **MRSTY** | Semantic types (e.g. *Disease or Syndrome*, *Pharmacologic Substance*) |
| **MRDEF** | Definition text, with the source vocabulary |
| **MRSAT** | Source-asserted attributes (mappings, codes, flags, etc.) |
| **MRHIER** | Hierarchy / context paths in each source vocabulary |

No external dependencies. Pure Python, streams each file once per query.

In [1]:
from pathlib import Path
from collections import defaultdict
from typing import Iterator

# === EDIT THIS ============================================================
# Folder that contains MRCONSO.RRF, MRSTY.RRF, MRDEF.RRF, MRSAT.RRF, MRHIER.RRF
UMLS_DIR = Path.home() / "Desktop"
# =========================================================================

# Column layouts straight from the UMLS Reference Manual.
RRF_COLUMNS = {
    "MRCONSO": ("CUI", "LAT", "TS", "LUI", "STT", "SUI", "ISPREF", "AUI",
                 "SAUI", "SCUI", "SDUI", "SAB", "TTY", "CODE", "STR",
                 "SRL", "SUPPRESS", "CVF"),
    "MRSTY":   ("CUI", "TUI", "STN", "STY", "ATUI", "CVF"),
    "MRDEF":   ("CUI", "AUI", "ATUI", "SATUI", "SAB", "DEF",
                 "SUPPRESS", "CVF"),
    "MRSAT":   ("CUI", "LUI", "SUI", "METAUI", "STYPE", "CODE", "ATUI",
                 "SATUI", "ATN", "SAB", "ATV", "SUPPRESS", "CVF"),
    "MRHIER":  ("CUI", "AUI", "CXN", "PAUI", "SAB", "RELA", "PTR", "HCD",
                 "CVF"),
}

def _table_path(name: str) -> Path:
    p = UMLS_DIR / f"{name}.RRF"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Edit UMLS_DIR above.")
    return p

def iter_rrf(name: str) -> Iterator[dict]:
    """Yield one dict per row of <name>.RRF. Streams; no memory blow-up."""
    cols = RRF_COLUMNS[name]
    path = _table_path(name)
    with path.open("r", encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").rstrip("|").split("|")
            if len(parts) < len(cols):
                parts += [""] * (len(cols) - len(parts))
            yield dict(zip(cols, parts))

def rows_for_cui(name: str, cui: str,
                 *, only_english: bool = True,
                 drop_suppressed: bool = True) -> list[dict]:
    """Return all rows of table <name> whose CUI matches.

    only_english   - MRCONSO only: keep only LAT=='ENG'
    drop_suppressed - drop rows where SUPPRESS != 'N' (if column present)
    """
    out = []
    for row in iter_rrf(name):
        if row["CUI"] != cui:
            continue
        if only_english and name == "MRCONSO" and row.get("LAT") != "ENG":
            continue
        if drop_suppressed and row.get("SUPPRESS", "N") not in ("N", ""):
            continue
        out.append(row)
    return out

In [2]:
# ---------- pretty-print helpers ----------

def _preferred_name(conso_rows: list[dict]) -> str | None:
    """Pick the canonical English preferred name from MRCONSO rows."""
    # ISPREF=Y, TS=P, STT=PF -> primary preferred atom.
    for r in conso_rows:
        if r.get("ISPREF") == "Y" and r.get("TS") == "P" and r.get("STT") == "PF":
            return r["STR"]
    return conso_rows[0]["STR"] if conso_rows else None

def _group_by(rows: list[dict], key: str) -> dict[str, list[dict]]:
    out: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        out[r.get(key, "")].append(r)
    return dict(out)

def show_concept(cui: str,
                 *, max_synonyms_per_sab: int = 5,
                 max_attrs_per_atn: int = 5,
                 max_hier_per_sab: int = 3) -> dict:
    """Look up everything about one CUI and print a human-readable summary.
    Returns the raw rows in a dict, so you can poke at them programmatically.
    """
    conso = rows_for_cui("MRCONSO", cui)
    sty   = rows_for_cui("MRSTY",   cui)
    defs  = rows_for_cui("MRDEF",   cui)
    sats  = rows_for_cui("MRSAT",   cui)
    hier  = rows_for_cui("MRHIER",  cui)

    pname = _preferred_name(conso)
    print(f"=== {cui}  {pname or '(no English name found)'} ===\n")

    # --- semantic types ---
    print("[Semantic types]  (MRSTY)")
    if not sty:
        print("  (none)")
    for r in sty:
        print(f"  {r['TUI']:>8}  {r['STY']:<40}  STN={r['STN']}")
    print()

    # --- definitions ---
    print("[Definitions]  (MRDEF)")
    if not defs:
        print("  (none)")
    for r in defs:
        print(f"  [{r['SAB']:<14}] {r['DEF']}")
    print()

    # --- synonyms grouped by source vocabulary ---
    print("[Atom strings]  (MRCONSO, English, non-suppressed)")
    by_sab = _group_by(conso, "SAB")
    for sab in sorted(by_sab):
        rows = by_sab[sab]
        seen = set()
        uniq = []
        for r in rows:
            if r["STR"] not in seen:
                seen.add(r["STR"])
                uniq.append(r)
            if len(uniq) >= max_synonyms_per_sab:
                break
        more = len({r['STR'] for r in rows}) - len(uniq)
        print(f"  [{sab}]  ({len({r['STR'] for r in rows})} unique strings)")
        for r in uniq:
            print(f"     TTY={r['TTY']:<6} CODE={r['CODE']:<14} {r['STR']}")
        if more > 0:
            print(f"     ... (+{more} more not shown)")
    print()

    # --- source-asserted attributes ---
    print("[Attributes]  (MRSAT, by attribute name)")
    if not sats:
        print("  (none)")
    by_atn = _group_by(sats, "ATN")
    for atn in sorted(by_atn):
        rows = by_atn[atn]
        print(f"  {atn}  ({len(rows)} values)")
        for r in rows[:max_attrs_per_atn]:
            print(f"     [{r['SAB']:<14}] {r['ATV']}")
        if len(rows) > max_attrs_per_atn:
            print(f"     ... (+{len(rows) - max_attrs_per_atn} more not shown)")
    print()

    # --- hierarchy paths ---
    print("[Hierarchy paths]  (MRHIER, PTR = path of parent AUIs)")
    if not hier:
        print("  (none)")
    by_sab_h = _group_by(hier, "SAB")
    for sab in sorted(by_sab_h):
        rows = by_sab_h[sab]
        print(f"  [{sab}]  ({len(rows)} paths)")
        for r in rows[:max_hier_per_sab]:
            ptr = r['PTR'] or '(root)'
            print(f"     RELA={r['RELA'] or '-':<10} {ptr}")
        if len(rows) > max_hier_per_sab:
            print(f"     ... (+{len(rows) - max_hier_per_sab} more not shown)")
    print()

    return {
        "cui": cui,
        "preferred_name": pname,
        "mrconso": conso,
        "mrsty":   sty,
        "mrdef":   defs,
        "mrsat":   sats,
        "mrhier":  hier,
    }

## Try it

Change `cui` to whatever you want to inspect. The first call will be slow because each table is scanned end-to-end (a few hundred MB to a few GB). For one-off lookups that's fine; for many lookups switch to an indexed loader.

In [4]:
info = show_concept("C0007561")   # Ceftriaxone

=== C0007561  ceftriaxone ===

[Semantic types]  (MRSTY)
      T109  Organic Chemical                          STN=A1.4.1.2.1
      T195  Antibiotic                                STN=A1.4.1.1.1.1

[Definitions]  (MRDEF)
  [MSH           ] A broad-spectrum cephalosporin antibiotic and cefotaxime derivative with a very long half-life and high penetrability to meninges, eyes and inner ears.
  [MSHSWE        ] Bredspektrumcefalosporin med lång halveringstid och god genomträngningsförmåga vid vanligtvis svåråtkomliga infektioner, som t ex sådana som berör hjärnhinnorna, ögonen, innerörat och urinvägarna.
  [NCI           ] A beta-lactam, third-generation cephalosporin antibiotic with bactericidal activity. Ceftriaxone binds to and inactivates penicillin-binding proteins (PBP) located on the inner membrane of the bacterial cell wall. PBPs participate in the terminal stages of assembling the bacterial cell wall, and in reshaping the cell wall during cell division. Inactivation of PBPs interf

In [ ]:
# Grab raw rows for further work:
info["mrdef"][:2]